# v4 — event decomposition

v3 values a swing with one regression: the run value of swinging at this pitch,
in this count. This notebook asks whether it is better to model what a swing
*produces* — a whiff, a foul, or a ball in play — and assemble the value from
those pieces:

```
Q_swing(s) = P(whiff | s)·RE(whiff, c) + P(foul | s)·RE(foul, c) + P(in play | s)·E[run value | in play, s]
```

The target is a function of (outcome, count), so both estimate the same
quantity. The premise is that whiffs are predictable (AUC 0.77 held out) while
the run value of a swing, pitch by pitch, is mostly luck.

**The comparison contract.** Decomposition is credited only against a *direct
twin* that shares everything else: the score (`signed_edge`), pitch features,
hitter priors, folds, the shared early-stopping learner, and the take model
itself — the twins call the same `fit_take`, and the notebook asserts their
`Q_take` is identical, so every difference comes from the swing branch.

**The selection rule** (as in v3): paired held-out MSE on the same swings
decides, with a 95% interval from resampling games — a tie keeps the simpler,
direct model; calibration checks the winner; the player-metric checks are
guardrails, reported with intervals, never deciding.

Held out throughout, on the personalized folds. 2026 is not loaded.

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import fit_direct, predict_both, V1_FEATURES
from src.decomposition import fit_decomposed

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
SEASONS = range(2021, 2026)          # 2026 is kept out of model selection; not loaded here
PF = E.PERSONALIZED_FOLDS
SHOW, MODEL = E.SHOW, E.MODEL_COLS
ACC_COLS = ['pitches', 'diff_%', 'diff_%_lo', 'diff_%_hi', 'verdict']
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']
PRIORS = ['prior_whiff', 'prior_foul']

def row(run, label, score=DEC.DEFAULT_SCORE):
    return E.harness(run, label, score)['summary']

def paired(a, b, labels):
    return E.paired_accuracy(a, b, labels)[ACC_COLS]

def rounds(run):
    return pd.DataFrame({season: m.rounds for season, m in run.models.items()}).T

In [2]:
df = D.drop_pitchers_batting(D.load_seasons(SEASONS, verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')

# Hitter priors, each season from the two seasons before it (2021 is prior data only).
key = ['game_pk', 'at_bat_number', 'pitch_number']
pdf = F.season_hot_zone(df).merge(F.season_contact_priors(df)[key + PRIORS],
                                  on=key, how='left', validate='one_to_one')
assert pdf[PRIORS].notna().all().all()
print(f'{len(pdf):,} pitches, seasons {sorted(pdf.season.unique())}')

2,827,989 pitches, seasons [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. B2 — generic twins

Full pitch frame, no hitter features in either twin. This isolates
decomposition itself.

In [3]:
direct = E.run_folds(pdf, V3_FEATURES, PF, fit=fit_direct)
decomp = E.run_folds(pdf, V3_FEATURES, PF, fit=fit_decomposed)
assert np.array_equal(direct.held.q_take.to_numpy(), decomp.held.q_take.to_numpy()), 'twins must share Q_take'
print('Q_take identical across twins: yes')
pd.concat({'direct': rounds(direct), 'decomposed': rounds(decomp)}, axis=1)

Q_take identical across twins: yes


direct       decomposed                  
       take swing       take outcome bip_value
2023    125    85        125     305        70
2024    155   105        155     535        69
2025    250   123        250     578        66

In [4]:
# The deciding test: squared error on the same swings. Negative = decomposed better.
paired(decomp, direct, ('decomposed', 'direct')).round(3)

pitches  diff_%  diff_%_lo  diff_%_hi            verdict
action season                                                          
take   2023     374424   0.000      0.000      0.000                tie
       2024     366046   0.000      0.000      0.000                tie
       2025     368391   0.000      0.000      0.000                tie
       pooled  1108861   0.000      0.000      0.000                tie
swing  2023     339205  -0.135     -0.176     -0.098  decomposed better
       2024     337782  -0.061     -0.089     -0.035  decomposed better
       2025     336884  -0.056     -0.081     -0.032  decomposed better
       pooled  1013871  -0.085     -0.103     -0.071  decomposed better

In [5]:
pd.concat({'direct': E.calibration_table(direct), 'decomposed': E.calibration_table(decomp)},
          names=['twin']).round(3)

slope  slope_lo  slope_hi
twin       action season                           
direct     take   2023    1.000     0.996     1.003
                  2024    0.998     0.994     1.002
                  2025    0.994     0.991     0.998
           swing  2023    1.111     1.086     1.134
                  2024    1.019     0.997     1.041
                  2025    1.030     1.008     1.054
decomposed take   2023    1.000     0.996     1.003
                  2024    0.998     0.994     1.002
                  2025    0.994     0.991     0.998
           swing  2023    1.087     1.063     1.112
                  2024    1.008     0.986     1.031
                  2025    1.038     1.015     1.061

In [6]:
# What the decomposed classifier gets right: whiff / foul / in play, held out.
E.outcome_diagnostics(decomp.held).round(4)

,swings,logloss,logloss_base_rate,whiff_auc,improvement_%,whiff_slope,foul_slope,bip_slope
season,,,,,,,,
2023,339205,0.9659,1.0851,0.7640,10.9846,1.0232,0.9591,1.0156
2024,337782,0.9617,1.0833,0.7675,11.2261,1.0078,0.9673,1.0052
2025,336884,0.9584,1.0831,0.7697,11.5097,1.0244,0.9927,1.0211


In [7]:
# How much do the two twins disagree, pitch by pitch?
a, b = direct.held, decomp.held
print(f"corr(Q_swing): {np.corrcoef(a.q_swing, b.q_swing)[0, 1]:.4f}   "
      f"corr(edge): {np.corrcoef(a.edge, b.edge)[0, 1]:.4f}")
print(f"recommended action differs on {(np.sign(a.edge) != np.sign(b.edge)).mean():.2%} of pitches")
sa, sb = E.player_metric(a, 'signed_edge'), E.player_metric(b, 'signed_edge')
m = sa.merge(sb, on=['season', 'batter'])
print(f"hitter-season decision values correlate {np.corrcoef(m.decision_value_x, m.decision_value_y)[0, 1]:.4f}")

corr(Q_swing): 0.9435   corr(edge): 0.9918
recommended action differs on 2.27% of pitches
hitter-season decision values correlate 0.9980


In [8]:
# Guardrails, decomposed minus direct, hitter-resampled intervals.
t2 = pd.DataFrame([row(direct, 'direct'), row(decomp, 'decomposed')]).set_index('variant')
display(t2[MODEL + SHOW])
E.metric_differences(decomp, direct, labels=('decomposed', 'direct')).round(3)

,swing MSE vs count-only %,swing pred slope,take MSE vs count-only %,take pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,,
direct,1.87,1.053,70.08,0.997,-0.916,0.711,0.815,0.584,0.269,0.104
decomposed,1.95,1.044,70.08,0.997,-0.914,0.706,0.816,0.583,0.270,0.100


,decomposed,direct,diff,diff_lo,diff_hi
check,,,,,
chase | zone_swing,-0.914,-0.916,0.002,0.000,0.003
zone_swing | chase,0.706,0.711,-0.005,-0.009,-0.000
YoY R2 (mean),0.583,0.584,-0.001,-0.008,0.005
Zone% |r|,0.270,0.269,0.001,-0.002,0.004
next-season r (partial),0.100,0.104,-0.004,-0.009,0.002


### Δ stability

Five refits of each twin on fold 3, each on a bootstrap sample of training
games, scored on the same 2025 pitches. A pitch is unstable if the recommended
action changes sign across refits. Same procedure for both twins.

In [9]:
def delta_stability(fit, features, frame, refits=5, seed=1126):
    train_seasons, valid = PF[-1]
    need = [f for f in features if f != 'count']
    rv = D.run_value_table(frame, train_seasons)
    part = D.apply_run_value(frame[frame.season.isin([*train_seasons, valid])].dropna(subset=need),
                             rv, name='target').dropna(subset=['target'])
    train, held = part[part.season.isin(train_seasons)], part[part.season == valid]
    by_game = train.groupby('game_pk').indices
    games = np.array(list(by_game))
    rng = np.random.default_rng(seed)
    edges = []
    for _ in range(refits):
        idx = np.concatenate([by_game[g] for g in rng.choice(games, len(games), replace=True)])
        model = fit(train.iloc[idx], features=features, rv=rv)
        edges.append(predict_both(held, model).edge.to_numpy())
    E_ = np.vstack(edges)
    unstable = (np.sign(E_) != np.sign(E_[0])).any(0)
    close = np.abs(E_.mean(0)) < 0.02
    return {'unstable, all pitches': unstable.mean(), 'unstable, |edge| < 0.02': unstable[close].mean(),
            'share of pitches with |edge| < 0.02': close.mean(), 'edge SD across refits': E_.std(0).mean()}

stab = pd.DataFrame({'direct': delta_stability(fit_direct, V3_FEATURES, pdf),
                     'decomposed': delta_stability(fit_decomposed, V3_FEATURES, pdf)})
stab.round(4)

,direct,decomposed
"unstable, all pitches",0.0647,0.0573
"unstable, |edge| < 0.02",0.4628,0.4554
share of pitches with |edge| < 0.02,0.1140,0.1144
edge SD across refits,0.0086,0.0070


**The decomposition is the better estimator.** On the same held-out swings its
squared error is 0.085% lower (interval 0.071–0.103%), and it wins in every
season (0.135%, 0.061%, 0.056%). The take side is identical by construction.
Its swing calibration is a little closer to 1 (1.01–1.09 against 1.02–1.11).

**It barely changes the score.** The twins recommend a different action on
2.3% of pitches and their hitter-season values correlate 0.998. Every guardrail
difference is within 0.005, and only zone-swing | chase (−0.005) sits just
outside its interval.

**And it is more stable.** Refit on resampled games, its Δ varies less (SD
0.0070 against 0.0086 runs) and fewer pitches change recommended action (5.7%
against 6.5%).

By the selection rule, **decomposition wins B2**: a more accurate estimate of
what a swing is worth, with the same metric.

## 2. B4 — personalized twins

Two hitter priors, each built from the two seasons before the one scored, with
the same smoothing and shrinkage as the hot zone:

- **damage** — `hot_zone`, the hitter's expected exit velocity by location;
- **contact** — `prior_whiff` and `prior_foul`, the hitter's whiff and foul
  rates per swing by location. New here.

In the decomposed twin each goes only to the sub-model it describes: contact
priors to the whiff / foul / in-play classifier, the hot zone to the in-play
value model. The direct twin gets both in its one swing regression. Neither
twin's take model sees them.

First, whether the contact priors are worth building at all: does a hitter's
surface reproduce the following season?

In [10]:
sw = F.swings_with_outcome(df)          # every season, 2021 included, as history
counts = sw.groupby(['season', 'batter']).size()
rows = []
for b in [2023, 2024, 2025]:
    keep = set(counts[b][counts[b] >= 300].index) & set(counts[b - 1][counts[b - 1] >= 300].index)
    target = F.surface_on_grid(F.value_surface(sw[(sw.season == b) & sw.batter.isin(keep)], ['_whiff', '_foul']))
    for label, window in [('one-season prior', [b - 1]), ('two-season prior', [b - 2, b - 1])]:
        prior = F.surface_on_grid(F.value_surface(sw[sw.season.isin(window) & sw.batter.isin(keep)], ['_whiff', '_foul']))
        common = sorted(set(target) & set(prior))
        T, P = np.stack([target[h] for h in common]), np.stack([prior[h] for h in common])
        for k, name in enumerate(['whiff', 'foul']):
            r = np.nanmean([np.corrcoef(T[:, n, k], P[:, n, k])[0, 1] for n in range(T.shape[1])])
            rows.append({'target season': b, 'prior': label, 'rate': name, 'hitters': len(common), 'r': r})
pd.DataFrame(rows).pivot_table(index=['rate', 'target season'], columns='prior', values='r').round(3)

prior                one-season prior  two-season prior
rate  target season                                    
foul  2023                      0.415             0.432
      2024                      0.422             0.457
      2025                      0.437             0.464
whiff 2023                      0.642             0.657
      2024                      0.646             0.669
      2025                      0.627             0.660

**Both contact priors reproduce, and a two-season window beats one.** A
hitter's whiff-rate surface predicts the next season's at r ≈ 0.66–0.67 (0.63–0.65
from one season) — about as stable as the hot zone. Foul rate is less stable,
r ≈ 0.43–0.46.

In [11]:
B4 = {
    'direct + hot zone':                      (fit_direct, V3_FEATURES + ['hot_zone']),
    'decomposed + hot zone':                  (fit_decomposed, V3_FEATURES + ['hot_zone']),
    'direct + hot zone + contact priors':     (fit_direct, V3_FEATURES + ['hot_zone'] + PRIORS),
    'decomposed + hot zone + contact priors': (fit_decomposed, V3_FEATURES + ['hot_zone'] + PRIORS),
}
b4 = {label: E.run_folds(pdf, feats, PF, fit=fit) for label, (fit, feats) in B4.items()}
for a_, b_ in [('direct + hot zone', 'decomposed + hot zone'),
               ('direct + hot zone + contact priors', 'decomposed + hot zone + contact priors')]:
    assert np.array_equal(b4[a_].held.q_take.to_numpy(), b4[b_].held.q_take.to_numpy()), 'twins must share Q_take'
t3 = pd.DataFrame([row(run, label) for label, run in b4.items()]).set_index('variant')
t3[MODEL + SHOW]

,swing MSE vs count-only %,swing pred slope,take MSE vs count-only %,take pred slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,,
direct + hot zone,1.93,1.025,70.08,0.997,-0.885,0.662,0.844,0.629,0.169,0.167
decomposed + hot zone,2.04,0.988,70.08,0.997,-0.877,0.648,0.845,0.632,0.162,0.171
direct + hot zone + contact priors,2.00,1.022,70.08,0.997,-0.878,0.641,0.847,0.624,0.174,0.164
decomposed + hot zone + contact priors,2.09,0.960,70.08,0.997,-0.862,0.582,0.847,0.615,0.184,0.173


In [12]:
pd.concat({
    'decomposed vs direct, + hot zone':
        paired(b4['decomposed + hot zone'], b4['direct + hot zone'], ('new', 'old')),
    'contact priors, direct':
        paired(b4['direct + hot zone + contact priors'], b4['direct + hot zone'], ('new', 'old')),
    'contact priors, decomposed':
        paired(b4['decomposed + hot zone + contact priors'], b4['decomposed + hot zone'], ('new', 'old')),
}, names=['comparison']).xs('pooled', level='season').round(3)

pitches  diff_%  diff_%_lo  diff_%_hi     verdict
comparison                       action                                                   
decomposed vs direct, + hot zone take    1108861   0.000      0.000      0.000         tie
                                 swing   1013871  -0.109     -0.129     -0.091  new better
contact priors, direct           take    1108861   0.000      0.000      0.000         tie
                                 swing   1013871  -0.073     -0.088     -0.056  new better
contact priors, decomposed       take    1108861   0.000      0.000      0.000         tie
                                 swing   1013871  -0.050     -0.065     -0.035  new better

In [13]:
pd.concat({label: E.calibration_table(b4[label]) for label in b4}, names=['variant']).xs('swing', level='action').round(3)

slope  slope_lo  slope_hi
variant                                season                           
direct + hot zone                      2023    1.054     1.026     1.077
                                       2024    1.009     0.988     1.031
                                       2025    1.012     0.990     1.034
decomposed + hot zone                  2023    0.975     0.951     0.997
                                       2024    0.989     0.968     1.010
                                       2025    0.999     0.978     1.022
direct + hot zone + contact priors     2023    1.073     1.049     1.095
                                       2024    0.991     0.971     1.014
                                       2025    1.003     0.983     1.025
decomposed + hot zone + contact priors 2023    0.936     0.913     0.958
                                       2024    0.966     0.946     0.987
                                       2025    0.978     0.958     1.001

In [14]:
pd.concat({label: E.outcome_diagnostics(b4[label].held) for label in
           ['decomposed + hot zone', 'decomposed + hot zone + contact priors']},
          names=['variant']).round(4)

swings  logloss  logloss_base_rate  whiff_auc  improvement_%  whiff_slope  foul_slope  bip_slope
variant                                season                                                                                                  
decomposed + hot zone                  2023    339205   0.9659             1.0851     0.7640        10.9846       1.0232      0.9591     1.0156
                                       2024    337782   0.9617             1.0833     0.7675        11.2261       1.0078      0.9673     1.0052
                                       2025    336884   0.9584             1.0831     0.7697        11.5097       1.0244      0.9927     1.0211
decomposed + hot zone + contact priors 2023    339205   0.9550             1.0851     0.7767        11.9921       0.9987      0.9323     0.9857
                                       2024    337782   0.9493             1.0833     0.7811        12.3670       0.9930      0.9554     0.9943
                                       2025    336884   0.9457             1.0831     0.7825        12.6811       1.0090      0.9750     1.0078

In [15]:
# Guardrails, hitter-resampled intervals.
pd.concat({
    'decomposed vs direct, + hot zone':
        E.metric_differences(b4['decomposed + hot zone'], b4['direct + hot zone'], labels=('new', 'old')),
    'contact priors, decomposed':
        E.metric_differences(b4['decomposed + hot zone + contact priors'], b4['decomposed + hot zone'], labels=('new', 'old')),
}, names=['comparison']).round(3)

new    old   diff  diff_lo  diff_hi
comparison                       check                                                         
decomposed vs direct, + hot zone chase | zone_swing      -0.877 -0.885  0.008    0.005    0.010
                                 zone_swing | chase       0.648  0.662 -0.014   -0.019   -0.009
                                 YoY R2 (mean)            0.632  0.629  0.002   -0.004    0.009
                                 Zone% |r|                0.162  0.169 -0.007   -0.012   -0.003
                                 next-season r (partial)  0.171  0.167  0.004   -0.001    0.011
contact priors, decomposed       chase | zone_swing      -0.862 -0.877  0.015    0.009    0.021
                                 zone_swing | chase       0.582  0.648 -0.066   -0.081   -0.051
                                 YoY R2 (mean)            0.615  0.632 -0.017   -0.028   -0.003
                                 Zone% |r|                0.184  0.162  0.022    0.008    0.037
                                 next-season r (partial)  0.173  0.171  0.002   -0.013    0.019

In [16]:
stab4 = pd.DataFrame({'direct + hot zone': delta_stability(fit_direct, V3_FEATURES + ['hot_zone'], pdf),
                      'decomposed + hot zone': delta_stability(fit_decomposed, V3_FEATURES + ['hot_zone'], pdf)})
stab4.round(4)

,direct + hot zone,decomposed + hot zone
"unstable, all pitches",0.0649,0.0575
"unstable, |edge| < 0.02",0.4666,0.4452
share of pitches with |edge| < 0.02,0.1156,0.1185
edge SD across refits,0.0087,0.0073


**Decomposition wins again once personalized.** With the hot zone, the
decomposed twin's swing MSE is 0.109% lower (0.091–0.129%). Its calibration is
closest to 1 of any model here (0.975–0.999, against 1.009–1.054 for the direct
twin). Guardrails: construct validity slips slightly (−0.877 / +0.648 against
−0.885 / +0.662, outside the intervals), Zone% improves (0.162 against 0.169),
and YoY and next-season validity tie.

**Contact priors make both twins more accurate.** Swing MSE is 0.073% lower in
the direct twin and 0.050% lower in the decomposed one, with intervals
excluding zero. The classifier improves too (whiff AUC 0.764–0.770 →
0.777–0.782).

**But the guardrails flag them.** In the decomposed twin they move the score
sharply: zone-swing | chase 0.648 → 0.582, YoY 0.632 → 0.615, Zone% 0.162 →
0.184, every interval excluding zero. They also over-spread the swing model
(slopes 0.936–0.978). A hitter who rarely whiffs gets a higher `Q_swing`
everywhere, so his swings score better whatever his decisions. Whether that is
a better personal valuation or a drift toward "is he a good contact hitter" is
exactly what the guardrails exist to flag.

**What the rule selects.** Accuracy says decomposed + hot zone + contact
priors. Its guardrail flag is large enough that the choice between it and
decomposed + hot zone is left open here, for discussion.

## 3. Against the benchmarks

Everything held out, 2023–2025. The two plate-discipline rates are computed
from the same pitches: O-Swing% (lower is better, so negated) and Z-Swing% −
O-Swing%. Their construct-validity cells are circular — they are built from
chase and zone-swing rates — so only reliability, contamination and prediction
are comparable. The current model is v3's selection, recalibrated as v3's
selection step decided.

In [17]:
CURRENT_RECALIBRATE = True          # v3's selection step recalibrated the chosen model
current = (E.run_folds(pdf, V3_FEATURES + ['hot_zone'], PF, fit=fit_direct, recalibrate=True)
           if CURRENT_RECALIBRATE else b4['direct + hot zone'])
v1_run = E.run_folds(pdf, V1_FEATURES, PF, fit=fit_direct)

h = v1_run.held
inz = D.in_rulebook_zone(h, ball_edge=True)
k_ = ['season', 'batter']
rates = pd.DataFrame({'o_swing': h[~inz].groupby(k_, observed=True).swing.mean(),
                      'z_swing': h[inz].groupby(k_, observed=True).swing.mean(),
                      'pitches': h.groupby(k_, observed=True).size()}).reset_index()
rates = rates[rates.pitches >= E.QUALIFY_PITCHES]

def as_scores(values):
    s = rates[k_ + ['pitches']].assign(raw=values)
    z = s.groupby('season').raw.transform(lambda v: (v - v.mean()) / v.std(ddof=0))
    return s.assign(decision_value=100 + 10 * z)

def checks_from_scores(scores, label):
    return {'variant': label,
            'YoY R2 (mean)': round(E.yoy_reliability(scores).r2.mean(), 3),
            'Zone% |r|': round(E.zone_pct_correlation(h, scores).r.abs().mean(), 3),
            'next-season r (partial)': round(E.predictive_validity(h, scores).r_partial.mean(), 3)}

bench = pd.DataFrame([
    {**E.metric_checks(v1_run.held, 'y_pred')['summary'], 'variant': 'v1 as designed (chosen_value)'},
    row(current, 'current model (v3 selection)'),
    row(b4['decomposed + hot zone'], 'decomposed + hot zone'),
    checks_from_scores(as_scores(-rates.o_swing), 'O-Swing% (negated)'),
    checks_from_scores(as_scores(rates.z_swing - rates.o_swing), 'Z-Swing% − O-Swing%'),
]).set_index('variant')
bench[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
v1 as designed (chosen_value),-0.890,0.678,0.763,0.538,0.116,0.129
current model (v3 selection),-0.887,0.660,0.844,0.630,0.172,0.166
decomposed + hot zone,-0.877,0.648,0.845,0.632,0.162,0.171
O-Swing% (negated),NaN,NaN,NaN,0.697,0.421,0.077
Z-Swing% − O-Swing%,NaN,NaN,NaN,0.557,0.114,0.146


## Findings

All figures held out, 2023–2025, on the personalized folds, under the shared
early-stopping learner. Models are compared by paired held-out accuracy; the
player checks are guardrails.

**The decomposition is a better estimator than the direct regression**, with
and without the hot zone (swing MSE 0.085% and 0.109% lower, every interval
excluding zero). It is also more stable and better calibrated, and it produces
almost the same metric (hitter-season values correlate 0.998). An earlier
version of this notebook called it a tie, because it compared the twins on the
player checks alone, which cannot see a difference in accuracy that leaves the
score unchanged.

**Contact priors are more accurate and flagged.** They improve both twins, but
in the decomposed twin they move the score away from plate discipline
(zone-swing | chase 0.648 → 0.582) and raise contamination. Whether to adopt
them is an open decision.

**Against simple benchmarks.** O-Swing% is heavily contaminated (Zone% |r|
0.421) and weakly predictive (0.077). Z-Swing% − O-Swing% is less contaminated
than the model (0.114 against 0.172) and nearly as predictive (0.146 against
0.166), though less reliable (YoY 0.557 against 0.630).